##  Ultimate Inventory Pipeline (Aligned & Safe)

This notebook ensures that the training and inference pipelines are perfectly aligned to prevent `feature_names mismatch` errors.

###  Pipeline Safety Measures:
1. **Explicit Feature List**: Features are defined once and exported.
2. **Feature Export**: Saves `inventory_features.pkl` as the single source of truth.
3. **Clean Training**: Removes all noise features (Region, Weather, etc.).

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import joblib
import os
import warnings

warnings.filterwarnings('ignore')
data_path = "../data/Processed/Inventory_management_cleaned.csv"
if not os.path.exists(data_path):
    data_path = "c:/Users/LENOVO/Downloads/Zidio project/RetailPulse/data/Processed/Inventory_management_cleaned.csv"

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store_ID_enc', 'Product_ID_enc', 'Date'])

In [2]:
def create_features(data):
    data = data.copy()
    group_cols = ['Store_ID_enc', 'Product_ID_enc']
    
    # Target: Next-day demand
    data['target'] = data.groupby(group_cols)['Units Sold'].shift(-1)
    
    # Lags (1, 7, 14, 30)
    for lag in [1, 7, 14, 30]:
        data[f'lag_{lag}'] = data.groupby(group_cols)['Units Sold'].shift(lag)
    
    # EMAs
    data['ema_7'] = data.groupby(group_cols)['Units Sold'].shift(1).transform(lambda x: x.ewm(span=7).mean())
    data['ema_30'] = data.groupby(group_cols)['Units Sold'].shift(1).transform(lambda x: x.ewm(span=30).mean())
    
    # Rolling Means
    for window in [7, 30]:
        data[f'rolling_mean_{window}'] = data.groupby(group_cols)['Units Sold'].shift(1).rolling(window).mean()
    
    # Cyclical Time
    data['month_sin'] = np.sin(2 * np.pi * data['Date'].dt.month / 12)
    data['month_cos'] = np.cos(2 * np.pi * data['Date'].dt.month / 12)
    data['dow_sin'] = np.sin(2 * np.pi * data['Date'].dt.dayofweek / 7)
    data['dow_cos'] = np.cos(2 * np.pi * data['Date'].dt.dayofweek / 7)
    
    return data.dropna(subset=['target']).dropna()

df_clean = create_features(df)

In [ ]:
# SINGLE SOURCE OF TRUTH
features = [
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'ema_7', 'ema_30',
    'rolling_mean_7', 'rolling_mean_30',
    'Price', 'Discount', 'Holiday/Promotion',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'Store_ID_enc', 'Product_ID_enc'
]

X = df_clean[features]
y = df_clean['target']

# Export feature list for Streamlit
joblib.dump(features, '../models/inventory_features.pkl')
print("Features exported to ../models/inventory_features.pkl")

Features exported to ../models/inventory_features.pkl


In [4]:
model = XGBRegressor(
    objective='reg:tweedie', 
    n_estimators=1000, 
    learning_rate=0.05, 
    max_depth=6
)
model.fit(X, y)

joblib.dump(model, '../models/inventory_model.pkl')
print("Model saved to ../models/inventory_model.pkl")